# Synova RAG - Build Vector Database

## Objective

This notebook builds the first Vector Database for Synova's Retrieval-Augmented Generation (RAG) system.

The workflow includes:

1. Load the multilingual knowledge base.
2. Convert JSON documents into retrievable text.
3. Split documents into semantic chunks.
4. Generate embeddings.
5. Build a FAISS vector database.
6. Save the index for inference.

This notebook validates the RAG pipeline using a small technical knowledge base before scaling to the full Synova documentation.

# 02. Import Libraries

In [1]:
import json
from pathlib import Path

import pandas as pd

# 03. Project Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

KB_DIR = PROJECT_ROOT / "knowledge_base"

VECTOR_DB_DIR = PROJECT_ROOT / "vector_db"

VECTOR_DB_DIR.mkdir(exist_ok=True)

# 04. Load JSON Files

In [3]:
json_files = list(KB_DIR.rglob("*.json"))

len(json_files)

4

In [4]:
json_files

[WindowsPath('c:/Users/balke/Desktop/AI SPIRE/Capstone Project/COPY/datapipeline/RAG/knowledge_base/technical/ar/01_password_and_account_access.json'),
 WindowsPath('c:/Users/balke/Desktop/AI SPIRE/Capstone Project/COPY/datapipeline/RAG/knowledge_base/technical/ar/vpn.json'),
 WindowsPath('c:/Users/balke/Desktop/AI SPIRE/Capstone Project/COPY/datapipeline/RAG/knowledge_base/technical/en/01_password_and_account_access.json'),
 WindowsPath('c:/Users/balke/Desktop/AI SPIRE/Capstone Project/COPY/datapipeline/RAG/knowledge_base/technical/en/vpn.json')]

# 05. Read Documents

In [5]:
documents = []

for file in json_files:
    with open(file, encoding="utf-8") as f:
        documents.append(json.load(f))

len(documents)

4

# 06. Quick Inspection

In [6]:
documents[0].keys()

dict_keys(['id', 'title', 'language', 'category', 'version', 'last_updated', 'tags', 'keywords', 'sections'])

# 07. Build Retriever Documents

In [8]:
# ما بدنا embed ال json
# بدنا نحوله ل نص غني بال metadata

rag_docs = []

for doc in documents:

    for section in doc["sections"]:

        if isinstance(section["content"], list):

            if len(section["content"]) > 0 and isinstance(section["content"][0], dict):

                text = "\n".join(
                    f"Q: {x['question']}\nA: {x['answer']}"
                    for x in section["content"]
                )

            else:

                text = "\n".join(section["content"])

        else:

            text = section["content"]

        rag_docs.append({
            "id": doc["id"],
            "title": doc["title"],
            "language": doc["language"],
            "category": doc["category"],
            "section": section["title"],
            "text": text
        })

# 08. DataFrame

In [11]:
rag_df = pd.DataFrame(rag_docs)

rag_df.head()

,id,title,language,category,section,text
0,tech_password_access,إعادة تعيين كلمة المرور والوصول إلى الحساب,ar,technical,نظرة عامة,تُعد مشكلات تسجيل الدخول والوصول إلى الحساب من...
1,tech_password_access,إعادة تعيين كلمة المرور والوصول إلى الحساب,ar,technical,المشكلات الشائعة,نسيان كلمة المرور\nقفل الحساب\nاسم مستخدم أو ك...
2,tech_password_access,إعادة تعيين كلمة المرور والوصول إلى الحساب,ar,technical,الأعراض,عدم القدرة على تسجيل الدخول\nظهور رسالة تفيد ب...
3,tech_password_access,إعادة تعيين كلمة المرور والوصول إلى الحساب,ar,technical,الأسباب المحتملة,إدخال كلمة مرور غير صحيحة\nانتهاء صلاحية كلمة ...
4,tech_password_access,إعادة تعيين كلمة المرور والوصول إلى الحساب,ar,technical,خطوات استكشاف المشكلة,تأكد من صحة اسم المستخدم.\nأعد تعيين كلمة المر...


# 09. Check Distribution

In [12]:
rag_df.groupby(["language","title"]).size()

language  title                                     
ar        إعادة تعيين كلمة المرور والوصول إلى الحساب    7
          الاتصال بالـ VPN واستكشاف المشكلات            7
en        Password Reset and Account Access             7
          VPN Connection and Troubleshooting            7
dtype: int64

# 10. Save

In [13]:
rag_df.to_csv(
    VECTOR_DB_DIR / "rag_documents.csv",
    index=False,
    encoding="utf-8-sig"
)